# CockroachDB CDC - Load Parquet Files with Column Family Merge

This notebook loads CockroachDB CDC Parquet files using Databricks Autoloader with automatic column family merging.

## Prerequisites
- Unity Catalog Volume with synced Parquet files
- Configuration files: `.env/cockroachdb_cdc_azure.json` and `.env/cockroachdb_pipelines.json`

## Step 1: Setup Configuration

In [1]:
import json
import os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

print("="*80)
print("CONFIGURATION SETUP")
print("="*80)

# Load configuration files
git_root = os.path.abspath("../../..")
cockroach_dir = f"{git_root}/sources/cockroachdb"
azure_json_path = f"{cockroach_dir}/.env/cockroachdb_cdc_azure.json"
pipeline_json_path = f"{cockroach_dir}/.env/cockroachdb_pipelines.json"

with open(azure_json_path, 'r') as f:
    azure_config = json.load(f)
with open(pipeline_json_path, 'r') as f:
    pipeline_config = json.load(f)

# Configuration
AZURE_STORAGE_ACCOUNT = azure_config["azure_storage_account"]
AZURE_STORAGE_KEY = azure_config["azure_storage_key"]
AZURE_CONTAINER = azure_config["azure_storage_container"]
PATH_PREFIX = pipeline_config["blob_prefix"]
SOURCE_TABLE = "usertable"
TARGET_CATALOG = pipeline_config["catalog"]
TARGET_SCHEMA = pipeline_config["schema"]
TARGET_TABLE = f"{SOURCE_TABLE}_delta"
TARGET_TABLE_PATH = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"
VOLUME_NAME = pipeline_config["volume_name"]
VOLUME_PATH = f"dbfs:/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{VOLUME_NAME}/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/"
CHECKPOINT_PATH = f"{VOLUME_PATH}/_checkpoints/{SOURCE_TABLE}"
PRIMARY_KEY_COLUMNS = ["ycsb_key"]

print(f"\n📋 Target: {TARGET_TABLE_PATH}")
print(f"📦 Volume: {VOLUME_PATH}")
print(f"🔑 Primary Key: {PRIMARY_KEY_COLUMNS}")
print("\n✅ Configuration loaded!")
print("="*80)

CONFIGURATION SETUP

📋 Target: main.robert_lee_cockroachdb.usertable_delta
📦 Volume: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/
🔑 Primary Key: ['ycsb_key']

✅ Configuration loaded!


## Step 2: Load with Autoloader

In [2]:
print("="*80)
print("STEP 2: LOAD PARQUET FILES WITH AUTOLOADER")
print("="*80)

df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useNotifications", "false")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", f"*{SOURCE_TABLE}*.parquet")
    .load(VOLUME_PATH)
)

print("✅ Autoloader configured")
print(f"📁 Source: {VOLUME_PATH}")

STEP 2: LOAD PARQUET FILES WITH AUTOLOADER
✅ Autoloader configured
📁 Source: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/


## Step 3: Add CDC Metadata

In [3]:
print("="*80)
print("STEP 3: TRANSFORM AND ENRICH CDC DATA")
print("="*80)

df_enriched = (df_raw
    .withColumn("_cdc_operation",
        F.when(F.col("__crdb__event_type") == "c", F.lit("UPSERT"))
         .when(F.col("__crdb__event_type") == "d", F.lit("DELETE"))
         .otherwise(F.lit("UNKNOWN"))
    )
    .withColumn("_cdc_timestamp", F.col("__crdb__updated").cast("string"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_processing_time", F.current_timestamp())
)

print("✅ CDC metadata added")

STEP 3: TRANSFORM AND ENRICH CDC DATA
✅ CDC metadata added


## Step 4: Merge Column Family Fragments

**CRITICAL**: Merges 11 column family fragments into 1 complete row per key.

In [4]:
import sys
import os
import importlib

parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import cockroachdb
importlib.reload(cockroachdb)
from cockroachdb import merge_column_family_fragments

print("="*80)
print("STEP 4: MERGE COLUMN FAMILY FRAGMENTS")
print("="*80)

df_merged = merge_column_family_fragments(
    df_enriched,
    primary_key_columns=PRIMARY_KEY_COLUMNS,
    debug=True
)

df_enriched = df_merged
print("\n✅ Merge complete!")
print("="*80)

STEP 4: MERGE COLUMN FAMILY FRAGMENTS

🔍 Column Family Merge (Streaming Mode)
   Primary key columns: ['ycsb_key']
   Data columns: 10 columns
     ['field9', 'field0', 'field1', 'field2', 'field3', 'field4', 'field5', 'field6', 'field7', 'field8']

🔧 Streaming mode: Applying merge
   (Cannot detect fragmentation in streaming DataFrames)
   - If column families exist: fragments will be merged
   - If no column families: merge is harmless no-op

✅ Merge transformation applied!
   Streaming DataFrame merged
   (Actual counts will be visible after writeStream completes)

✅ Merge complete!


## Step 5: Clear Checkpoint (Optional)

Only run if re-processing. Skip on first run.

In [5]:
# ============================================================================
# FAST CHECKPOINT CLEARING (Parallel Deletion)
# ============================================================================

import cockroachdb
importlib.reload(cockroachdb)
from cockroachdb import parallel_delete_checkpoint

print("="*80)
print("FAST CHECKPOINT CLEARING")
print("="*80)

# Delete checkpoint directory in parallel (5-20x faster!)
result = parallel_delete_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    dbutils=dbutils,  # Pass notebook's dbutils context
    max_workers=20,   # Number of parallel threads
    debug=True        # Show progress
)

# Drop the Delta table
try:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_PATH}")
    print(f"\n✅ Dropped table: {TARGET_TABLE_PATH}")
except Exception as e:
    print(f"ℹ️  Table drop: {e}")

print()
print("="*80)
print("CLEAR COMPLETE")
print("="*80)
print(f"⏱️  Total time: {result['elapsed_seconds']:.1f}s")
print(f"📊 Items deleted: {result['deleted_count']}")

if result['failed_count'] > 0:
    print(f"⚠️  Failed deletions: {result['failed_count']}")
    
print("="*80)



FAST CHECKPOINT CLEARING
PARALLEL CHECKPOINT DELETION
Path: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files//_checkpoints/usertable
Workers: 20

📊 Found:
   Subdirectories: 1
   Files: 0

🔍 Recursively scanning directories (max 3 levels deep)...
   schema/: 1 leaf directories

   Total: 1 directories to delete

🔥 Deleting 1 directories in parallel...
   ✅ _schemas/0

   Deleted 1 directories

🧹 Cleaning up parent directory...
   ✅ Deleted parent: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files//_checkpoints/usertable

DELETION SUMMARY
✅ Successfully deleted: 1 items
⏱️  Time: 7.6s

✅ Dropped table: main.robert_lee_cockroachdb.usertable_delta

CLEAR COMPLETE
⏱️  Total time: 7.6s
📊 Items deleted: 1


## Step 6: Write to Delta Table

**Expected**: ~9,995 rows (not 109,945!)

In [6]:
print("="*80)
print("STEP 6: WRITE TO DELTA TABLE")
print("="*80)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
print(f"✅ Schema: {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Write stream using complete mode (supports streaming aggregations!)
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("complete")  # ✅ Works with streaming aggregations
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/delta")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_PATH)
)

print(f"\n🚀 Writing to: {TARGET_TABLE_PATH}")
print(f"📍 Checkpoint: {CHECKPOINT_PATH}/delta")
print(f"⚙️  Output mode: complete (supports aggregations)")
print(f"⚙️  Spark version: {spark.version}")
print("⏳ Processing with merge applied...")

# Wait for completion
query.awaitTermination()

print("\n✅ Write complete!")
print("💡 Table now contains merged rows (not fragments)")  
print("="*80)

STEP 6: WRITE TO DELTA TABLE
✅ Schema: main.robert_lee_cockroachdb

🚀 Writing to: main.robert_lee_cockroachdb.usertable_delta
📍 Checkpoint: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files//_checkpoints/usertable/delta
⚙️  Output mode: complete (supports aggregations)
⚙️  Spark version: 4.1.0
⏳ Processing with merge applied...

✅ Write complete!
💡 Table now contains merged rows (not fragments)


## Step 7: Verify Results

In [7]:
print("="*80)
print("STEP 7: VERIFY")
print("="*80)

df_delta = spark.table(TARGET_TABLE_PATH)
total_count = df_delta.count()

print(f"\n📊 Delta table: {total_count:,} rows")

if total_count < 20000:
    print("   ✅ Merge worked!")
else:
    print("   ⚠️  Too many rows (merge failed?)")

print("\n📊 Operation Breakdown:")
display(df_delta.groupBy("_cdc_operation").count())

print("\n📋 Sample:")
display(df_delta.limit(10))

STEP 7: VERIFY

📊 Delta table: 9,995 rows
   ✅ Merge worked!

📊 Operation Breakdown:


,_cdc_operation,count
0,UPSERT,9995



📋 Sample:


,ycsb_key,field9,field0,field1,field2,field3,field4,field5,field6,field7,field8,__crdb__event_type,__crdb__updated,_rescued_data,_cdc_operation,_cdc_timestamp,_source_file,_processing_time
0,user10391469795925176447,SCwlVoYNkpvBIGMmALrwbiKiBFgIGkdpEDHKBobVsnDjQUSJGhCCPydsrZlKjBeYGjiZkSLAPpPlXxLjDfBxixKgtjeHUbTJSziI,BrlyNLQAZDmcfyKsKkeKXngJbMSYwrADwSFbzFAbRaeQgtLItmJgndMSXkKMJrEAIdQZYkETowIPwjrQdcYCKmxUrDFIKWaSETbR,rprtpeMcSoqSyYvAQbEVdcVxWeaPHeqcIbGQtcWsuiuXKzdsTlhYUGnArwvRgrJENgVqfXsPztFdcNTQvCGSYRCgWUfrtrXNAomO,RweZQLiwBHhaLbwRTkJqGXvVHokriYItnGOmSjOcEnsoKyxvALauWhHOpFBZlfhVDXTdDGQHmIjyUiwqyPaPHirnyfMuxwbabexy,fDmJxcTBwnuXsqYAHCeUrTTyNPuNINOkbwJwfFIksibDKtkJIAotJeeMMhuFzBlqeYvpMWngqRZSsCDGMSYdYQJsJuivOxpiqGpL,excGxacRSmAOgLiAqrfwYwAYQWidnEfRdJMCfoyZmLSswhrdJRbdmouWnwdsSTyvXrtqIyeCbKDfDRtjZbOOiGCYeBoPAjmJhZsY,sEGtiZXtQDqjffEWOpeGnDuvYSsGBxyrnnRMntpuNnhSxSympxCniHDUnLYdZKLcZmrjlqDsVWrgZiSvFNcCXtjbonEkFQgEnJxA,tqHGtOogfOqCCzheJLuzlrCdgYlodJcVyxaIDuwUgqEMAISnpSdZeyfeBgPsmstuvRyMKrbFfpjBcRDCOekORqLaAWioEhorhKrZ,meIXMtYNCImyDpkBQiqGQrLVmDIIwRSpeDrrFbtfkneqFQLXnROfDJyFpRqZvWOLHdAycrgXGqWllDXEupXVFUqkXHtrspyoCyJx,kPOQVAwsTyeSkiQKsxVvqMnnRXFPkZRoxsKFkHIiYSNhqAXzVOKHJuMSBlLcPAIZkZaxkYinOCbfaHinaCXOPLbImYhaLuIWYzBe,c,1766164464280983190.0000000000,None,UPSERT,1766164464280983190.0000000000,dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/202512191714242809831900000000000-d340a6adc87c635b-1-375-00000007-usertable+fam_6_field5-4.parquet,2026-01-06 20:19:43.529
1,user1079925162761333720,zZJrTSVRGKAoVzpjHuCGtmsBalxQJtoYWyWokxSgafvagXbKDDQtmYANjOTGEVRoDcBjLLJPciRLuhIMKRrjVXSEBFvGBPyiwafb,mzkseyQJBFpqTiRHPKqweQZvvGJUXmQhyHMLLCxkMbhOEjHIJGCpPcRigzPzfPCdgMaHeMKHpSoxTthmMhSLmnAfGySTIXBpxrcu,HCVlInTOuDeXpkovOVFAWPcggFVLxApMkwiJxmctJHfoQxEOXoIKDYjqtKDcVhnYVMEogATmzBlLGWeUMkTHRQqMdifARmApYKNw,WgTOlNiVtQZELciZwbBInKBWDgPNLQmJcEAOyfuxmZjxIppcGueZRynCwkALESjdtZWfHAIwVtAijBfFpgySiAFliYvSugEfpdaG,dtrKAEmowtBcmhdtExLIWVzEEZjHKHSylgDLOLNwRmOzHMFjOYEIykhTLrNInqriOeEdfJphVQnwezigQGecQdoxUOxdwnUYEtBQ,lEpYHsltJuErAoBFqqDAKHNLvoRnNbhsxdkLuSsHkHIdjEbYbxelSpenmwFynPyDVkrMujRWdQhAenqdrESOfSujxNTApsoeTXvv,dPbxIgRXzBsYQTYagUayOVEdJwgncBIcdkdHoAnXohclAfVTCaaLGYjROFtUchmVcDhprpiEuQhGpJopLTRMLKsHCfEYzeFYXHiW,YBOAhFKjlutEftZSMKzUyrwUSEddqcslDCbzwdcrnKimfGvdxBIoVlqiKUZQdKuiSwrTLqnhfhcctWqjdZFjmorvZIWMzrHFRGcf,QViqCuegBtfYAEgEcpSJixKWacgNOXOwRwnqlyFfZtFmTXYSwkAyzgdSrCFMLkNMSrBuJxWSgmDTwwxgAYZNvAXCTudLhUUMfkhI,vgJjlCHwGytXlwcIPEASBDJFxKgteTQCSKxtSJnxhEKzOuRjswEEnrJUrLdhiREZrtoslqRAXdAaPZWbJEVQRoXxEJGzCxeKTnDk,c,1766164464280983190.0000000000,None,UPSERT,1766164464280983190.0000000000,dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/202512191714242809831900000000000-d340a6adc87c635b-1-375-00000007-usertable+fam_6_field5-4.parquet,2026-01-06 20:19:43.529
2,user10854806712176560338,ajehAazchtUWBxOColMyIuAYBTUteLyUQxsbehwcJoxmqRGOfORjmlRfoYDcsdhjhypNnaivRaaVaZCuiuydgMFYyMgQtpjqjGoz,mSptauOZXRcjITTNTbwmETYkdyPZOCFqCizlQGlxTszUZFmkhvkICNtOVmNNRvyRoTxFcGuNqYxKYpIQGgzHjqHBrGBTCsIpYMUZ,ikHLasZXYGGcBblGqVbvXVhMUZfPbSpJkKMjvEyCqLFcFsPfubzUNlhinOxnszioFsJuFYHKJXqyIWujzSOAgFIwFYqHiOlbHhPb,PjvqGQdwrtkYMlhjludOdxOHSuNHsMyWvQsHOmhKWAxEmFzHoeqJDsuTzQBkxtSBpfnbAcNqKFXHmuhHDnEcDwqwVcOCatyILjpT,IfepRRvNwkPXfgFFpDguvlvcZrflHxSzFnfCAXEaeayHpdVxninSEslbyZwATtjDXjqupvnUrWNZcjccdxqvzcgLhCLaMnJBxiKL,UOEioSCtxbJITzIUAxzIdNowCexkVlyUtsNHasJAbpQgGjSfubRUhEvJgnDYCLbBmxZnnQSaxocNrMSdULTrqvbctjoYIvsEenlu,XvMZsmJjBIfOtFdBJhPrKhEjQEWTHCQBIjkyPaJWjogfnAFgxnOtteetOhgXhzVpxROkLEYCFNfobWEZmJUsCGWkngRrgjwARlTT,SHHMwwZhUpXAnnEaxJTbggYJuZMPjpCSAWWvdUrgBDOmeFrBukUqFtznTgfwRafBAjmKPqACSXjGNhFfVZlfYXlPBmiVrvJCniTg,nWsjRhGWilFNdLwanePRwoILzAwGRMPYeuxSXQonofHIGIszAioydBflUdjfyqwsbcgGnyxkaajPpPbMeWQpicpqClRjFvCrLSuE,yQpGimDKbTiSwvOFYURQrZdopUMYsehuolwCnqBQSYhIdjIRyNgWktKuXIBYVoXkhRncJnSvAGvGCUuyIXwJNwChHQIJfwhdXswe,c,1766164464280983190.0000000000,None,UPSERT,1766

## Step 8: Compare with Source Files

In [8]:
import sys, os, importlib

parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import cockroachdb
importlib.reload(cockroachdb)
from cockroachdb import analyze_volume_changefeed_files

print("="*80)
print("STEP 8: VERIFY SOURCE FILES")
print("="*80)

raw_stats = analyze_volume_changefeed_files(volume_path=VOLUME_PATH, debug=False)

print(f"\n📊 Volume files: {raw_stats['file_count']}")
print(f"   Unique keys: {raw_stats.get('unique_keys', 'N/A')}")
print(f"   UPSERT: {raw_stats['snapshot']:,}")
print(f"   DELETE: {raw_stats['delete']:,}")
print(f"   Total: {raw_stats['snapshot'] + raw_stats['delete']:,}")

print(f"\n📊 Comparison:")
print(f"   Delta: {total_count:,}")
print(f"   Volume: {raw_stats['snapshot'] + raw_stats['delete']:,}")

if total_count == raw_stats['snapshot'] + raw_stats['delete']:
    print(f"\n   ✅✅✅ PERFECT MATCH! ✅✅✅")
    print(f"   Column family merge worked correctly!")
else:
    diff = total_count - (raw_stats['snapshot'] + raw_stats['delete'])
    print(f"\n   ⚠️  MISMATCH: {diff:,} difference")

STEP 8: VERIFY SOURCE FILES

📊 Volume files: 11
   Unique keys: 9995
   UPSERT: 9,995
   DELETE: 0
   Total: 9,995

📊 Comparison:
   Delta: 9,995
   Volume: 9,995

   ✅✅✅ PERFECT MATCH! ✅✅✅
   Column family merge worked correctly!


## Summary

✅ **Successfully loaded with column family merge!**

### Results
- Delta table: ~9,995 rows (merged)
- Volume files: ~9,995 rows (deduplicated)
- Match: YES ✅

### What Happened
1. Autoloader read 11 Parquet files (1 per column family)
2. Merge function grouped by primary key
3. 109,945 fragments → 9,995 complete rows
4. Delta table has correct count!

### Next Steps
- Schedule for incremental updates
- Query with time travel
- Monitor changefeed health